# 08 — V-JEPA action probe (Fase 0, opcional)

**Objetivo:** Embedding / clasificación sobre clip segmentado (I+D post-MVP PRD).

**Nota:** Requiere pesos HuggingFace y GPU opcional. Si falla, estado `SKIPPED` controlado.


## Prerrequisitos

**02** clips; código en `models/vjepa2-main/`.


## 1. Setup


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

from _common.io import (
    ensure_scripts_on_path,
    read_json,
    repo_root,
    setup_logging,
    stage_output_dir,
    write_json,
)
from loguru import logger

ensure_scripts_on_path()
setup_logging()

VJEPA_ROOT = repo_root() / "models" / "vjepa2-main"
if VJEPA_ROOT.is_dir() and str(VJEPA_ROOT) not in sys.path:
    sys.path.insert(0, str(VJEPA_ROOT))


## 2. Configuration


In [2]:
OUT_DIR = stage_output_dir("08_vjepa")
SEGMENTS_MANIFEST = stage_output_dir("02_segments") / "manifest.json"
CLIP_ID = "clip_0000"
HF_MODEL_ID = "facebook/vjepa2-vitg-fpc64-256-ssv2"
MAX_FRAMES = 16
SKIPPED = False
STATUS = "pending"


## 3. Cargar frames del clip


In [3]:
import cv2

manifest = read_json(SEGMENTS_MANIFEST)
clip_entry = next((c for c in manifest["clips"] if c["clip_id"] == CLIP_ID), manifest["clips"][0])
clip_dir = repo_root() / clip_entry["path"]
paths = sorted(clip_dir.glob("frame_*.jpg"))[:MAX_FRAMES]
if not paths:
    raise FileNotFoundError(f"Sin frames en {clip_dir}")

frames = [cv2.imread(str(p)) for p in paths]
frames = [f for f in frames if f is not None]
logger.info("Frames cargados: {}", len(frames))


21:59:28 | INFO | Frames cargados: 16


## 4. Inferencia V-JEPA (HuggingFace) o fallback


In [4]:
embedding = None
probe_result = {"clip_id": clip_entry["clip_id"], "num_frames": len(frames)}

try:
    import torch
    from transformers import AutoModel, AutoVideoProcessor

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoVideoProcessor.from_pretrained(HF_MODEL_ID)
    model = AutoModel.from_pretrained(HF_MODEL_ID).to(device).eval()

    # T x H x W x C (RGB)
    rgb = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames]
    video = np.stack(rgb, axis=0)
    inputs = processor(list(video), return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        feats = model.get_vision_features(**inputs)
    embedding = feats.detach().cpu().numpy()
    probe_result.update({
        "status": "ok",
        "backend": "huggingface",
        "model_id": HF_MODEL_ID,
        "embedding_shape": list(embedding.shape),
        "device": device,
    })
except Exception as exc:
    SKIPPED = True
    embedding = np.zeros((1, 32), dtype=np.float32)
    probe_result.update({
        "status": "SKIPPED",
        "reason": str(exc),
        "hint": "Descargue pesos HF o use GPU; ver models/vjepa2-main/README.md",
    })
    logger.warning("V-JEPA skip: {}", exc)

np.save(OUT_DIR / "embedding.npy", embedding)
write_json(OUT_DIR / "probe_result.json", probe_result)
probe_result


21:59:28 | WARNING | V-JEPA skip: No module named 'transformers'


{'clip_id': 'clip_0000',
 'num_frames': 16,
 'status': 'SKIPPED',
 'reason': "No module named 'transformers'",
 'hint': 'Descargue pesos HF o use GPU; ver models/vjepa2-main/README.md'}

## 5. Validación


In [5]:
assert (OUT_DIR / "embedding.npy").is_file()
assert probe_result.get("status") in ("ok", "SKIPPED")
print(f"V-JEPA probe: {probe_result['status']}")


V-JEPA probe: SKIPPED
